# Maintained strategy composition

This notebook exercises deterministic maintained calculations without an event store or broker.

In [ ]:
from datetime import UTC, datetime, timedelta
from trader import Bar
from trader_standard import SmaIndicator, SmaCrossoverSignal

now = datetime(2026, 1, 1, tzinfo=UTC)
closes = (12.0, 10.0, 10.0, 11.0)
bars = tuple(
    Bar(ts=now - timedelta(hours=index), open=value, high=value, low=value, close=value, volume=1.0, vwap=None, trade_count=None)
    for index, value in enumerate(closes)
)
short = SmaIndicator(period=2)
long = SmaIndicator(period=3)
signal = SmaCrossoverSignal(short=short, long=long)
assert short.compute_series(bars)[0] == 11.0
assert long.compute_series(bars)[0] == 32.0 / 3.0
assert signal.compute(bars) == 1.0

In [ ]:
from trader_standard import build_trend_following_strategy

strategy = build_trend_following_strategy(
    symbols=("AAPL", "MSFT"),
    asset_class="stocks",
    timeframe="1Hour",
    target_qty_when_long=1.0,
)
assert strategy.strategy_id == "trend_following"
assert strategy.strategy_info.parameters["symbols"] == ["AAPL", "MSFT"]